# Multi-Model Conversations

**Week 2 Day 1 - Learning Lab**

Creating conversations between multiple chatbots with different personalities.

## Intent

Learn to:
- Build conversation history with message structure
- Create multi-model conversations (2-way, 3-way, 4-way)
- Manage conversation state across multiple models
- Understand system prompts for personality definition

## Expected Insights

- Message structure: system (personality) + user/assistant (history)
- How to build conversation history incrementally
- Pattern for multi-agent systems


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion
from IPython.display import Markdown, display

load_dotenv(override=True)

# Setup clients
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
deepseek_client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com") if deepseek_api_key else None

# Check Ollama
import requests
try:
    requests.get("http://localhost:11434/", timeout=2)
    ollama_available = True
    ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
except:
    ollama_available = False
    ollama_client = None

print("Clients initialized")


## 2-Way Conversation: GPT vs Gemini


In [ ]:
# Define models and personalities
gpt_model = "gpt-4o-mini"
gemini_model = "gemini/gemini-2.5-flash-lite"

gpt_system = "You are a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way."

gemini_system = "You are a very polite, courteous chatbot. You try to agree with everything the other person says, or find common ground. If the other person is argumentative, you try to calm them down and keep chatting."

# Initialize conversation
gpt_messages = ["Hi there"]
gemini_messages = ["Hi"]

# Display initial messages
display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))

# Conversation functions
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, gemini in zip(gpt_messages, gemini_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": gemini})
    response = openai_client.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    for gpt, gemini_msg in zip(gpt_messages, gemini_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": gemini_msg})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = completion(model=gemini_model, messages=messages)
    return response.choices[0].message.content

# Run a few rounds
for i in range(3):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)


## 4-Way Conversation: GPT, Gemini, Ollama, DeepSeek

(Only if all providers are available)


In [ ]:
# Only run if all providers available
if deepseek_client and ollama_available and ollama_client:
    # Define all models and personalities
    ollama_model = "llama3.2"
    deepseek_model = "deepseek-chat"
    
    ollama_system = "You are an analytical, curious chatbot. You ask thoughtful questions, seek to understand different perspectives, and provide balanced, well-reasoned responses."
    
    deepseek_system = "You are a pragmatic, solution-oriented chatbot. You focus on facts, offer practical insights, and help move conversations toward actionable conclusions."
    
    # Reset conversation
    gpt_messages = ["Hi there"]
    gemini_messages = ["Hi"]
    ollama_messages = ["Hello"]
    deepseek_messages = ["Hey"]
    
    # Display initial messages
    display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
    display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))
    display(Markdown(f"### Ollama:\n{ollama_messages[0]}\n"))
    display(Markdown(f"### DeepSeek:\n{deepseek_messages[0]}\n"))
    
    # 4-way conversation functions
    def call_gpt_4way():
        messages = [{"role": "system", "content": gpt_system}]
        min_len = min(len(gpt_messages), len(gemini_messages), len(ollama_messages), len(deepseek_messages))
        for i in range(min_len - 1):
            messages.append({"role": "assistant", "content": gpt_messages[i]})
            messages.append({"role": "user", "content": f"Gemini: {gemini_messages[i]}"})
            messages.append({"role": "user", "content": f"Ollama: {ollama_messages[i]}"})
            messages.append({"role": "user", "content": f"DeepSeek: {deepseek_messages[i]}"})
        messages.append({"role": "user", "content": f"Gemini: {gemini_messages[-1]}"})
        messages.append({"role": "user", "content": f"Ollama: {ollama_messages[-1]}"})
        messages.append({"role": "user", "content": f"DeepSeek: {deepseek_messages[-1]}"})
        response = openai_client.chat.completions.create(model=gpt_model, messages=messages)
        return response.choices[0].message.content
    
    def call_gemini_4way():
        messages = [{"role": "system", "content": gemini_system}]
        min_len = min(len(gpt_messages), len(gemini_messages), len(ollama_messages), len(deepseek_messages))
        for i in range(min_len - 1):
            messages.append({"role": "user", "content": f"GPT: {gpt_messages[i]}"})
            messages.append({"role": "user", "content": f"Ollama: {ollama_messages[i]}"})
            messages.append({"role": "user", "content": f"DeepSeek: {deepseek_messages[i]}"})
            messages.append({"role": "assistant", "content": gemini_messages[i]})
        messages.append({"role": "user", "content": f"GPT: {gpt_messages[-1]}"})
        messages.append({"role": "user", "content": f"Ollama: {ollama_messages[-1]}"})
        messages.append({"role": "user", "content": f"DeepSeek: {deepseek_messages[-1]}"})
        response = completion(model=gemini_model, messages=messages)
        return response.choices[0].message.content
    
    def call_ollama_4way():
        messages = [{"role": "system", "content": ollama_system}]
        min_len = min(len(gpt_messages), len(gemini_messages), len(ollama_messages), len(deepseek_messages))
        for i in range(min_len - 1):
            messages.append({"role": "user", "content": f"GPT: {gpt_messages[i]}"})
            messages.append({"role": "user", "content": f"Gemini: {gemini_messages[i]}"})
            messages.append({"role": "user", "content": f"DeepSeek: {deepseek_messages[i]}"})
            messages.append({"role": "assistant", "content": ollama_messages[i]})
        messages.append({"role": "user", "content": f"GPT: {gpt_messages[-1]}"})
        messages.append({"role": "user", "content": f"Gemini: {gemini_messages[-1]}"})
        messages.append({"role": "user", "content": f"DeepSeek: {deepseek_messages[-1]}"})
        response = ollama_client.chat.completions.create(model=ollama_model, messages=messages)
        return response.choices[0].message.content
    
    def call_deepseek_4way():
        messages = [{"role": "system", "content": deepseek_system}]
        min_len = min(len(gpt_messages), len(gemini_messages), len(ollama_messages), len(deepseek_messages))
        for i in range(min_len - 1):
            messages.append({"role": "user", "content": f"GPT: {gpt_messages[i]}"})
            messages.append({"role": "user", "content": f"Gemini: {gemini_messages[i]}"})
            messages.append({"role": "user", "content": f"Ollama: {ollama_messages[i]}"})
            messages.append({"role": "assistant", "content": deepseek_messages[i]})
        messages.append({"role": "user", "content": f"GPT: {gpt_messages[-1]}"})
        messages.append({"role": "user", "content": f"Gemini: {gemini_messages[-1]}"})
        messages.append({"role": "user", "content": f"Ollama: {ollama_messages[-1]}"})
        response = deepseek_client.chat.completions.create(model=deepseek_model, messages=messages)
        return response.choices[0].message.content
    
    # Run one round
    print("Running one round of 4-way conversation...\n")
    
    gpt_next = call_gpt_4way()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    gemini_next = call_gemini_4way()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)
    
    ollama_next = call_ollama_4way()
    display(Markdown(f"### Ollama:\n{ollama_next}\n"))
    ollama_messages.append(ollama_next)
    
    deepseek_next = call_deepseek_4way()
    display(Markdown(f"### DeepSeek:\n{deepseek_next}\n"))
    deepseek_messages.append(deepseek_next)
else:
    print("⚠️  Not all providers available. Need: DeepSeek API key, Ollama running locally")


## Key Takeaways

1. **Message structure:** `[{"role": "system/user/assistant", "content": "..."}]`
2. **System prompt:** Defines personality/role
3. **Conversation history:** Each model needs full history to maintain context
4. **Pattern:** Build incrementally, each model sees all previous exchanges
5. **Business relevance:** Foundation for conversational AI assistants and multi-agent systems
